In [0]:
base = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/"
display(dbutils.fs.ls(base + "optimized_chunks/"))
display(dbutils.fs.ls(base + "optimized_embeddings/DMRetriever-33M/"))


In [0]:
base = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/"

df_chunks = (spark.read.json(base + "optimized_chunks/*/chunks.jsonl")
  .select("chunk_id", "source_file", "chunk_index_in_file", "text")
  .repartition(256)          # tune 128-512
  .persist()
)

df_chunks.count()            # materialize once


In [0]:
import json, io
import numpy as np
import pandas as pd
from pyspark.sql.functions import col, regexp_extract
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType

# 1) Batch read all embeddings.npy bytes (1 Spark job)
df_npy = (spark.read.format("binaryFile")
          .load(base + "optimized_embeddings/DMRetriever-33M/*/embeddings.npy")
          .withColumn("source_dir", regexp_extract(col("path"), r"/DMRetriever-33M/([^/]+)/embeddings\.npy$", 1))
          .select("source_dir", col("content").alias("npy_bytes")))

# 2) Batch read all chunk_ids.json bytes (1 Spark job)
df_ids = (spark.read.format("binaryFile")
          .load(base + "optimized_embeddings/DMRetriever-33M/*/chunk_ids.json")
          .withColumn("source_dir", regexp_extract(col("path"), r"/DMRetriever-33M/([^/]+)/chunk_ids\.json$", 1))
          .select("source_dir", col("content").alias("ids_bytes")))

# 3) Join pairs (1 Spark job)
df_pair = (df_npy.join(df_ids, on="source_dir", how="inner")
           .select("source_dir", "npy_bytes", "ids_bytes"))

# Optional: quick stage progress
n_sources = df_pair.count()
print("paired_sources =", n_sources)

schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("embedding", ArrayType(FloatType()), False),
])

def decode_pairs(pdf_iter):
    # Decode inside executors (parallel)
    for pdf in pdf_iter:
        out = []
        for _, r in pdf.iterrows():
            ids = json.loads(bytes(r["ids_bytes"]).decode("utf-8"))
            vecs = np.load(io.BytesIO(bytes(r["npy_bytes"])))  # [n, dim]
            out.extend((cid, v.astype("float32").tolist()) for cid, v in zip(ids, vecs))
        yield pd.DataFrame(out, columns=["chunk_id", "embedding"])

# 4) Distributed decode
df_emb = df_pair.mapInPandas(decode_pairs, schema=schema)

# Optional: materialize to speed later join (writes/reads can help)
# df_emb = df_emb.repartition(200).cache()
# df_emb.count()


In [0]:
df_emb.write \
    .mode("overwrite") \
    .saveAsTable("tdis_dev_data_catalog.tdir.optimized_embeddings_dmretriever33m")


In [0]:
df_chunks = (
    spark.read.json(base + "optimized_chunks/*/chunks.jsonl")
    .select("chunk_id", "source_file", "chunk_index_in_file", "text")
)

df_chunks.write \
    .mode("overwrite") \
    .saveAsTable("tdis_dev_data_catalog.tdir.optimized_chunks_text")


In [0]:
df_final = df_chunks.join(df_emb, on="chunk_id", how="inner")

df_final.write \
    .mode("overwrite") \
    .saveAsTable("tdis_dev_data_catalog.tdir.optimized_rag_chunks")


In [0]:
print("chunks_text:", spark.table("tdis_dev_data_catalog.tdir.optimized_chunks_text").count())
print("embeddings:", spark.table("tdis_dev_data_catalog.tdir.optimized_embeddings_dmretriever33m").count())
print("rag_chunks:", spark.table("tdis_dev_data_catalog.tdir.optimized_rag_chunks").count())


In [0]:
from pyspark.sql.functions import countDistinct, count

spark.table("tdis_dev_data_catalog.tdir.optimized_embeddings_dmretriever33m") \
    .select(count("*").alias("rows"),
            countDistinct("chunk_id").alias("unique_chunk_id")) \
    .show()
